## Load Dimension & Fact Tables into Hospital_Datawarehouse

This notebook reads the 18 healthcare CSV files from the Bronze Lakehouse, infers
the schema for each table, creates the corresponding tables in the
**Hospital_Datawarehouse**, and loads the data.

| Layer | Tables | Count |
|-------|--------|-------|
| **Dimensions** | dim_nurses, dim_patients, dim_hospital_units, dim_documentation_types, dim_medications, dim_diagnoses | 6 |
| **Facts (batch)** | fact_shifts, fact_patient_encounters, fact_care_plans, fact_burnout_surveys, fact_patient_satisfaction, fact_documentation_quality | 6 |
| **Facts (events)** | fact_documentation_events, fact_medication_administration, fact_vital_signs, fact_assessments, fact_ehr_interactions, fact_handoff_reports | 6 |

> **Prerequisites:**
> 1. Upload all CSV files from `data/` to the **Hospital_Data_Bronze** Lakehouse → `Files/healthcare_data/`
> 2. Attach **Hospital_Data_Bronze** as the default Lakehouse for this notebook
> 3. Ensure the **Hospital_Datawarehouse** exists in the **CentralData** workspace

In [ ]:
# ── Configuration ───────────────────────────────────────────────────────
WAREHOUSE_NAME   = "Hospital_Datawarehouse"   # Fabric Data Warehouse name
SCHEMA_NAME      = "dbo"                       # Target schema
CSV_BASE_PATH    = "Files/healthcare_data"     # Path inside the attached Lakehouse

# Tables to load — maps CSV base name → warehouse table name
DIMENSION_TABLES = {
    "dim_nurses":               "dim_nurses",
    "dim_patients":             "dim_patients",
    "dim_hospital_units":       "dim_hospital_units",
    "dim_documentation_types":  "dim_documentation_types",
    "dim_medications":          "dim_medications",
    "dim_diagnoses":            "dim_diagnoses",
}

FACT_TABLES = {
    "fact_shifts":                   "fact_shifts",
    "fact_patient_encounters":       "fact_patient_encounters",
    "fact_care_plans":               "fact_care_plans",
    "fact_burnout_surveys":          "fact_burnout_surveys",
    "fact_patient_satisfaction":     "fact_patient_satisfaction",
    "fact_documentation_quality":    "fact_documentation_quality",
    "fact_documentation_events":     "fact_documentation_events",
    "fact_medication_administration":"fact_medication_administration",
    "fact_vital_signs":              "fact_vital_signs",
    "fact_assessments":              "fact_assessments",
    "fact_ehr_interactions":         "fact_ehr_interactions",
    "fact_handoff_reports":          "fact_handoff_reports",
}

ALL_TABLES = {**DIMENSION_TABLES, **FACT_TABLES}
print(f"Tables to load: {len(ALL_TABLES)} ({len(DIMENSION_TABLES)} dim + {len(FACT_TABLES)} fact)")

In [ ]:
# ── Spark type → T-SQL type mapping ─────────────────────────────────────
from pyspark.sql.types import (
    StringType, IntegerType, LongType, FloatType, DoubleType,
    BooleanType, DateType, TimestampType, DecimalType,
)

SPARK_TO_SQL = {
    StringType():    "NVARCHAR(4000)",
    IntegerType():   "INT",
    LongType():      "BIGINT",
    FloatType():     "FLOAT",
    DoubleType():    "FLOAT",
    BooleanType():   "BIT",
    DateType():      "DATE",
    TimestampType(): "DATETIME2",
}


def spark_type_to_sql(spark_type):
    """Convert a Spark data type to the best T-SQL equivalent."""
    if isinstance(spark_type, DecimalType):
        return f"DECIMAL({spark_type.precision},{spark_type.scale})"
    return SPARK_TO_SQL.get(spark_type, "NVARCHAR(4000)")


def generate_create_table_sql(df, full_table_name):
    """Generate a CREATE TABLE statement from a Spark DataFrame schema."""
    col_defs = []
    for field in df.schema.fields:
        sql_type = spark_type_to_sql(field.dataType)
        nullable = "NULL" if field.nullable else "NOT NULL"
        col_defs.append(f"    [{field.name}] {sql_type} {nullable}")
    cols_sql = ",\n".join(col_defs)
    return f"CREATE TABLE {full_table_name} (\n{cols_sql}\n)"


print("Helper functions ready.")

In [ ]:
# ── Read CSVs, create warehouse tables, and load data ────────────────
results = []

for csv_name, table_name in ALL_TABLES.items():
    full_table = f"{WAREHOUSE_NAME}.{SCHEMA_NAME}.{table_name}"
    csv_path   = f"{CSV_BASE_PATH}/{csv_name}.csv"

    print(f"\n{'─'*60}")
    print(f"Processing: {csv_name} → {full_table}")

    # 1. Read CSV with schema inference
    df = (spark.read
          .option("header", True)
          .option("inferSchema", True)
          .csv(csv_path))
    row_count = df.count()
    print(f"  Read {row_count} rows, {len(df.columns)} columns")

    # 2. Generate and display the inferred CREATE TABLE DDL
    ddl = generate_create_table_sql(df, full_table)
    print(f"  DDL:\n{ddl[:200]}..." if len(ddl) > 200 else f"  DDL:\n{ddl}")

    # 3. Drop table if it already exists, then create + load
    try:
        spark.sql(f"DROP TABLE IF EXISTS {full_table}")
        spark.sql(ddl)
        df.write.mode("append").synapsesql(full_table)
        status = "✓"
        print(f"  ✓ Loaded {row_count} rows")
    except Exception as e:
        status = f"✗ {e}"
        print(f"  ✗ Error: {e}")

    results.append((table_name, row_count, len(df.columns), status))

print(f"\n{'═'*60}")
print(f"Done — processed {len(results)} tables.")

In [ ]:
# ── Summary ──────────────────────────────────────────────────────────
import pandas as pd

summary = pd.DataFrame(results, columns=["Table", "Rows", "Columns", "Status"])
summary["Layer"] = summary["Table"].apply(
    lambda t: "Dimension" if t.startswith("dim_") else "Fact"
)

print(f"\n{'═'*60}")
print(f" Hospital_Datawarehouse — Load Report")
print(f"{'═'*60}")
print(f" Dimensions : {len(summary[summary.Layer=='Dimension'])} tables")
print(f" Facts      : {len(summary[summary.Layer=='Fact'])} tables")
print(f" Total rows : {summary.Rows.sum()}")
print(f"{'═'*60}\n")

display(summary[["Layer", "Table", "Rows", "Columns", "Status"]])

### Verification — Query the Warehouse

After running this notebook, verify the tables exist in **Hospital_Datawarehouse**:

```sql
-- List all loaded tables
SELECT TABLE_SCHEMA, TABLE_NAME, TABLE_TYPE
FROM Hospital_Datawarehouse.INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'dbo'
ORDER BY TABLE_NAME;

-- Quick row counts
SELECT 'dim_nurses' AS tbl, COUNT(*) AS cnt FROM dbo.dim_nurses
UNION ALL SELECT 'dim_patients', COUNT(*) FROM dbo.dim_patients
UNION ALL SELECT 'fact_documentation_events', COUNT(*) FROM dbo.fact_documentation_events
UNION ALL SELECT 'fact_shifts', COUNT(*) FROM dbo.fact_shifts;

-- Documentation burden per nurse (warehouse SQL)
SELECT n.first_name + ' ' + n.last_name AS nurse_name,
       n.role, n.unit_id,
       SUM(d.duration_minutes) AS total_doc_minutes,
       COUNT(*) AS doc_event_count
FROM dbo.dim_nurses n
JOIN dbo.fact_documentation_events d ON n.nurse_id = d.nurse_id
GROUP BY n.first_name, n.last_name, n.role, n.unit_id
ORDER BY total_doc_minutes DESC;
```